# Pre-entrega 6: delegación multi-agente con LangGraph

Este notebook ejecuta, paso a paso, el mismo orquestador del repositorio:

```text
START → Supervisor → Investigador → Supervisor → Analista → Supervisor → Síntesis → Supervisor → END
```

Tiene dos modos:

| Modo | ¿Qué decide? | ¿Necesita API key? | ¿Consume tokens? |
|---|---|---|---|
| **Offline** (por defecto) | Un modelo determinista local (`OfflineDemoModel`) | No | No |
| **OpenAI** | `ChatOpenAI` | Sí | Sí |

En ambos modos se ejecutan **de verdad** el grafo, las herramientas, las validaciones Pydantic y la persistencia en SQLite. Lo único que cambia es quién toma las decisiones.

> La búsqueda de mercado es **simulada**: las cifras (120 y 144 millones de USD) sirven para evaluar la arquitectura, no son datos reales.

## 1. Preparar el entorno

- **Google Colab:** la celda clona el repositorio e instala el paquete.
- **Jupyter local:** abrí el notebook desde la carpeta del proyecto con el entorno virtual activado (`python -m pip install -e ".[notebook]"`).

Además se agrega `src/` al `sys.path`, así el paquete se puede importar sin reiniciar el kernel después de instalarlo.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rguima-ai/langgraph-multi-agent-orchestrator.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content/langgraph-multi-agent-orchestrator")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
else:
    # En local el notebook vive en notebooks/: la raíz del proyecto es la carpeta padre.
    ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print(f"Proyecto: {ROOT}")

## 2. Elegir el modo y cargar la API key (solo modo OpenAI)

Dejá `USE_OPENAI = False` para la demo sin costo. Para usar OpenAI:

- **Colab:** guardá `OPENAI_API_KEY` en **Secrets** (ícono de llave) y habilitá el acceso del notebook.
- **Local:** copiá `.env.example` como `.env` y completá la clave.

Nunca escribas la clave dentro de una celda: quedaría guardada en el notebook. Esta celda tampoco la imprime.

In [ ]:
import os

USE_OPENAI = False

if USE_OPENAI:
    if IN_COLAB:
        from google.colab import userdata

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    else:
        from dotenv import load_dotenv

        load_dotenv(ROOT / ".env")
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY (Secrets de Colab o archivo .env).")
    print("API key cargada (no se muestra).")
else:
    print("Modo offline: no se usa ninguna API key.")

## 3. Construir el equipo

Los cuatro nodos reciben el **mismo modelo**, pero cada uno tiene su propio rol:

| Nodo | Cómo usa el modelo | Escribe en el estado |
|---|---|---|
| Supervisor | `with_structured_output(SupervisorDecision)` | `next_agent`, `quality_feedback`, `step_count` |
| Investigador | `bind_tools([search_market_data])` | `research_output` |
| Analista | `bind_tools([calculate_percentage_change])` | `analysis_output` |
| Síntesis | `with_structured_output(FinalReport)` | `final_report` |

Cada especialista escribe **solo su parcela** (aislamiento de escritura). Las listas `messages`, `contributions` y `errors` se acumulan mediante *reducers*.

El diagrama Mermaid se genera desde el grafo compilado: las flechas punteadas son las **aristas condicionales** del Supervisor.

In [ ]:
from multi_agent_orchestrator.graph import build_graph
from multi_agent_orchestrator.main import build_team
from multi_agent_orchestrator.offline import OfflineDemoModel

if USE_OPENAI:
    from langchain_openai import ChatOpenAI

    model = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"), temperature=0)
else:
    model = OfflineDemoModel()

team = build_team(model)
print(build_graph(team).get_graph().draw_mermaid())

## 4. Estado inicial

El pedido obliga a combinar dos dominios (investigación y cálculo) antes de sintetizar. `max_steps` es el freno contra bucles: el Supervisor puede tomar como máximo 8 decisiones.

In [ ]:
from multi_agent_orchestrator.main import DEMO_REQUEST, initial_state

MAX_STEPS = 8
state = initial_state(DEMO_REQUEST, MAX_STEPS)
print(state["request"])
print("Campos del estado:", sorted(state))

## 5. Ejecutar con streaming y persistencia

- `AsyncSqliteSaver` guarda un checkpoint del estado después de cada nodo en `data/notebook_checkpoints.sqlite` (ignorado por Git).
- `thread_id` identifica la conversación: con el mismo id se puede retomar el hilo.
- `recursion_limit` es el límite de LangGraph; se calcula a partir de `MAX_STEPS` para que el freno propio del Supervisor actúe primero.
- `astream(..., stream_mode="updates")` muestra qué actualizó cada nodo **mientras ocurre**.

In [ ]:
from datetime import datetime

from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

DB_PATH = ROOT / "data" / "notebook_checkpoints.sqlite"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
config = {
    "configurable": {"thread_id": f"notebook-{datetime.now():%Y%m%d-%H%M%S}"},
    "recursion_limit": MAX_STEPS * 2 + 2,
}

async with AsyncSqliteSaver.from_conn_string(str(DB_PATH)) as checkpointer:
    graph = build_graph(team, checkpointer=checkpointer)
    async for update in graph.astream(state, config=config, stream_mode="updates"):
        for node, values in update.items():
            if node == "supervisor":
                print(f"[{values['step_count']}] Supervisor → {values['next_agent']}: "
                      f"{values['supervisor_reason']}")
            else:
                status = values["contributions"][0]["status"]
                print(f"    {node}: {status}")
    snapshot = await graph.aget_state(config)

final_state = snapshot.values

## 6. Informe final

Síntesis solo usa la investigación y el análisis ya validados: no busca datos nuevos. Además, el informe debe citar al menos un `source_id` real de la investigación.

In [ ]:
from pprint import pprint

print("Tarea completada:", final_state["task_completed"])
print("Variación calculada:", final_state["analysis_output"]["result"], "%")
pprint(final_state["final_report"], sort_dicts=False, width=100)

## 7. Auditoría: quién aportó qué

`contributions` registra cada intervención y `errors` los fallos recuperables. Esta es la información que se guarda en la traza: **no** contiene prompts ni razonamiento privado del modelo, solo decisiones y artefactos estructurados.

In [ ]:
for position, item in enumerate(final_state["contributions"], start=1):
    print(f"{position}. {item['agent']:<12} {item['status']:<9} {item.get('summary', '')}")

print("\nErrores registrados:", final_state["errors"] or "ninguno")

## 8. Recuperar el hilo desde SQLite

Un checkpointer **nuevo**, abierto sobre el mismo archivo, recupera el estado con el mismo `thread_id`. Eso demuestra que la persistencia no depende de la memoria del proceso.

In [ ]:
async with AsyncSqliteSaver.from_conn_string(str(DB_PATH)) as checkpointer:
    restored_graph = build_graph(team, checkpointer=checkpointer)
    restored = await restored_graph.aget_state(config)
    history = [item async for item in restored_graph.aget_state_history(config)]

print("Thread:", config["configurable"]["thread_id"])
print("Decisiones del Supervisor:", restored.values["step_count"])
print("Checkpoints guardados:", len(history))

## 9. Para experimentar

- Cambiá `MAX_STEPS` a 4 y observá que el flujo completo entra justo en el límite.
- En modo OpenAI, el Supervisor puede devolver trabajo a un especialista con `quality_feedback` si el artefacto no cumple la rúbrica.
- Las pruebas del repositorio (`python -m pytest -q`) cubren refinamiento, timeouts, división por cero y el límite de pasos sin consumir tokens.